# 04 - NFIP pool simulation

Thin caller over `nfip.simulation`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))   # the nfip package
sys.path.insert(0, os.path.abspath('src'))       # existing ST_Cluster.py, if used
import numpy as np, pandas as pd
from src import data, preprocess, config
from src.simulation import run_simulation
!pip install openpyxl

## Parameters

In [ ]:
export = True; save = True; plot = False; pls_print = False
test_case = 'block'        # 'historic', 'block' or 'random'
premium_type = '_Discount'           # '_Full', '_Discount' or ''
use_reinsurance = True
base_case = '' if use_reinsurance else '_base_case'

if test_case == 'historic':
    simulations = 1; steps = 100; block_size = 10
else:
    steps = 100; block_size = 10; simulations = 1000

## Data load

In [ ]:
gdf_counties = data.load_counties()
gdf_states   = data.load_states()
gdf_rivers   = data.load_rivers()
gdf_urban    = data.load_urban()

In [ ]:
clustered_claims, optimal_cluster = data.load_clustered_claims()
clustered_claims = preprocess.add_claim_fields(clustered_claims)
cpi = data.load_cpi_annual()
clustered_claims = preprocess.cpi_adjust_claims(clustered_claims, cpi)

In [ ]:
enso = data.load_enso_annual()   # available if you condition coupons on ENSO

## Premiums + state balance sheet

In [ ]:
aggregated_risk_policies = data.load_premiums(premium_type)
state_claims = data.load_state_claims()
gdf_states, total_premium = preprocess.build_state_balance_sheet(
    gdf_states, aggregated_risk_policies, state_claims)
print('total premium:', total_premium)

## Reinsurance/ILS Coverage Flag + Run

In [ ]:
clustered_claims = preprocess.annotate_is_ts(claims=clustered_claims, optimal_cluster=optimal_cluster)

In [ ]:
out = run_simulation(
    clustered_claims, gdf_states, optimal_cluster,
    test_case=test_case, steps=steps, block_size=block_size, simulations=simulations,
    use_reinsurance=use_reinsurance, pls_print=pls_print)
cluster_state_df      = out['cluster_state_df']
state_balance_df      = out['state_balance_df']
final_balances_df     = out['final_balances_df']
balance_transition_df = out['balance_transition_df']

In [ ]:
if export:
    tag = f'{test_case}_new{premium_type}{base_case}'
    cluster_state_df.to_csv(f'Results/cluster_state_{tag}.csv')
    state_balance_df.to_csv(f'Results/state_balance_{tag}.csv')
    final_balances_df.to_csv(f'Results/final_balances_{tag}.csv')
    balance_transition_df.to_csv(f'Results/balance_transition_{tag}.csv')